# The dictionary carries values in the graph

Inspect the dictionary and a field that refers to it.

## Prepare the model

Run this cell first. It imports the required types and defines the model used by this example. Start with a fresh repository Python kernel so declarations from other examples do not enter this model.

In [1]:
from pydantic import Field

from aoa.action_machine.domain import BaseCoreEntity, BaseEnumEntity, value
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entity import entity


class ShopDomain(BaseDomain):
    """Group the shop declarations."""

    name = "shop"
    description = "Shop model"


@entity(description="Currencies accepted by the shop", domain=ShopDomain)
class CurrencyEnumEntity(BaseEnumEntity):
    """Declare the shop's currency vocabulary."""

    symbol: str = Field(description="Currency symbol")

    EUR = value(code="eur", display_name="Euro", symbol="€")
    USD = value(code="usd", display_name="US dollar", symbol="$")


@entity(description="Price tag", domain=ShopDomain)
class PriceTagCoreEntity(BaseCoreEntity):
    """Associate an amount with its currency."""

    id: str = Field(description="Tag identifier")
    amount: float = Field(description="Price amount")
    currency: CurrencyEnumEntity = Field(description="Money unit")

## Run the example

The following cell performs the operation described above. Its assertions check the result; any expected refusal is caught explicitly.

In [2]:
from aoa.action_machine.graph.nodes.entity_graph_node import EntityGraphNode

dictionary = EntityGraphNode(CurrencyEnumEntity)
tag = EntityGraphNode(PriceTagCoreEntity)
field = next(edge.target_node for edge in tag.entity_field_edges if edge.target_node.label == "currency")
print("Kind:", dictionary.properties["entity_kind"])
print("Values:", [value["code"] for value in dictionary.properties["values"]])
print("Reference:", field.properties["reference_qualname"].split(".")[-1])

Kind: enum
Values: ['eur', 'usd']
Reference: CurrencyEnumEntity


## What the result means

The graph describes one dictionary entity with a list of declared values. The price’s currency field points to that dictionary through `reference_qualname`. Individual currencies are not separate entity nodes, and building this graph does not read price rows.